# Prueba de los pesos de matrículas lejanas sobre el vídeo

Usa los pesos `best.pt` de `matriculas_yolo11n_lejanas` (generados con `generar_placas_lejanas.py` y `entrenar_lejanas.py`) sobre el vídeo `C0142.MP4`, a resolución completa.

In [1]:
import cv2
from pathlib import Path

from ultralytics import YOLO

In [2]:
# Pesos entrenados con matrículas reducidas (P4/runs/matriculas_yolo11n_lejanas)
weights = Path("runs/matriculas_yolo11n_lejanas/weights/best.pt")
filename = "../datos/videos/C0142.MP4"
IMGSZ = 1920  # resolución completa del vídeo, para no reducir las matrículas lejanas
CONF = 0.25   # umbral de confianza, más bajo = más detecciones lejanas (y más falsos positivos)

assert weights.exists(), f"No se encuentran los pesos: {weights.resolve()}"

model = YOLO(str(weights))
print(model.names)

{0: 'License Plate'}


Vista rápida con las utilidades de ultralytics, sin filtro de vehículos y sin guardar nada (pulsa `q` o cierra la ventana para parar). Se puede saltar.

In [3]:
# stream=True procesa fotograma a fotograma, sin acumular los resultados en la RAM
for _ in model(filename, stream=True, show=True, imgsz=IMGSZ, conf=CONF, verbose=False):
    pass

cv2.destroyAllWindows()

Genera el vídeo final en `videos/salida_C0142_lejanas.mp4` (unos 5 minutos con GPU). Solo dibuja las matrículas que caen dentro de un vehículo, lo que descarta falsos positivos del fondo. Si pulsas `q`, el vídeo se corta en ese punto.

In [4]:
salida = "../videos/salida_C0142_lejanas.mp4"

detector_vehiculos = YOLO("yolo11n.pt")
VEHICULOS = [2, 3, 5, 7]  # coche, moto, autobús, camión


def dentro_de_vehiculo(caja, vehiculos):
    """Acepta la caja si tiene forma de matrícula y está en la parte baja de un vehículo."""
    x1, y1, x2, y2 = caja
    ancho, alto = x2 - x1, y2 - y1
    if alto <= 0 or not 1.2 <= ancho / alto <= 6.5:
        return False
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    for vx1, vy1, vx2, vy2 in vehiculos:
        v_ancho, v_alto = vx2 - vx1, vy2 - vy1
        if (vx1 <= cx <= vx2 and vy1 + 0.35 * v_alto <= cy <= vy2 + 0.05 * v_alto
                and ancho <= 0.65 * v_ancho):
            return True
    return False


vid = cv2.VideoCapture(filename)
fps = vid.get(cv2.CAP_PROP_FPS) or 25
w = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(salida, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

while True:
    ret, img = vid.read()
    if not ret:
        break

    vehiculos = detector_vehiculos(img, imgsz=IMGSZ, conf=0.15, classes=VEHICULOS,
                                   verbose=False)[0].boxes.xyxy.cpu().numpy()
    placas = model(img, imgsz=IMGSZ, conf=CONF, verbose=False)[0].boxes

    for caja, conf in zip(placas.xyxy.cpu().numpy(), placas.conf.cpu().numpy()):
        if not dentro_de_vehiculo(caja, vehiculos):
            continue
        x1, y1, x2, y2 = map(int, caja)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 255), 2)
        cv2.putText(img, f"Matricula {conf:.2f}", (x1, max(y1 - 8, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)

    out.write(img)
    cv2.imshow("Matriculas", img)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

vid.release()
out.release()
cv2.destroyAllWindows()
print("Vídeo guardado en", salida)

Vídeo guardado en ../videos/salida_C0142_lejanas.mp4
